In [18]:
# ==========================================
# GroupDNA - WhatsApp Chat Analyzer
# Week 1 Minor Project
# Name :Suhasini Mallick
# Batch: Data Analysis
# Date: 28 june 2026
# ==========================================

In [19]:
import numpy as np
from datetime import datetime

In [20]:
with open("hostel_bois.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total lines in file :", len(lines))

Total lines in file : 3178


In [21]:
messages = []

system_count = 0
media_count = 0
deleted_count = 0

## **FEATURE 1** - Parser

In [22]:
# Read WhatsApp chat file
# Parse date, time, sender and message
# Store all messages

for line in lines:

    line = line.strip()

    if line == "":
        continue

    if " - " not in line:
        continue

    date_time, data = line.split(" - ", 1)

    if ":" not in data:
        system_count += 1
        continue

    sender, message = data.split(":", 1)

    sender = sender.strip()
    message = message.strip()

    if message == "<Media omitted>":
        media_count += 1

    if message == "This message was deleted":
        deleted_count += 1

    messages.append({
        "datetime": date_time,
        "sender": sender,
        "message": message
    })

In [23]:
print("Total Parsed Messages :", len(messages))
print("System Messages :", system_count)
print("Media Messages :", media_count)
print("Deleted Messages :", deleted_count)

Total Parsed Messages : 3174
System Messages : 4
Media Messages : 32
Deleted Messages : 15


In [24]:
for i in range(5):
    print(messages[i])

{'datetime': '01/04/24, 01:17', 'sender': 'Rahul', 'message': 'scene fix'}
{'datetime': '01/04/24, 01:17', 'sender': 'Rahul', 'message': 'haan'}
{'datetime': '01/04/24, 01:18', 'sender': 'Rahul', 'message': 'kya scene'}
{'datetime': '01/04/24, 02:13', 'sender': 'Rahul', 'message': 'abhi free hai?'}
{'datetime': '01/04/24, 02:13', 'sender': 'Rahul', 'message': 'abey'}


## **FEATURE 2** - Group Overview

In [25]:
message_count = {}
participants = set()
dates = []

In [26]:
# Count total messages
# Count messages by each person
# Display group overview

for msg in messages:

    sender = msg["sender"]

    participants.add(sender)

    if sender in message_count:
        message_count[sender] += 1
    else:
        message_count[sender] = 1

    date = msg["datetime"].split(",")[0]
    dates.append(date)

In [27]:
start_date = datetime.strptime(dates[0], "%d/%m/%y")
end_date = datetime.strptime(dates[-1], "%d/%m/%y")

total_days = (end_date - start_date).days + 1

In [28]:
print("=" * 60)
print("GROUP OVERVIEW")
print("=" * 60)

print("Group Name      : Hostel Bois 4ever")
print("Period          :", start_date.strftime("%d %B %Y"),
      "to", end_date.strftime("%d %B %Y"))
print("Total Days      :", total_days)
print("Total Messages  :", len(messages))
print("Participants    :", len(participants))

print("\nMESSAGES PER PERSON")
print("-" * 60)
sorted_people = sorted(message_count.items(),
                       key=lambda x: x[1],
                       reverse=True)

for person, count in sorted_people:

    percentage = (count / len(messages)) * 100

    print(f"{person:<10} : {count:>4} ({percentage:.1f}%)")

GROUP OVERVIEW
Group Name      : Hostel Bois 4ever
Period          : 01 April 2024 to 30 May 2024
Total Days      : 60
Total Messages  : 3174
Participants    : 6

MESSAGES PER PERSON
------------------------------------------------------------
Rahul      :  953 (30.0%)
Priya      :  718 (22.6%)
Neha       :  635 (20.0%)
Aman       :  490 (15.4%)
Karan      :  354 (11.2%)
Vikas      :   24 (0.8%)


## **FEATURE 3** - Activity Analysis

In [29]:
day_count = {}
hour_count = {}

In [30]:

for msg in messages:

    date_time = msg["datetime"]

    date = date_time.split(",")[0]

    time = date_time.split(",")[1].strip()

    hour = time.split(":")[0]

    # Count messages per day
    if date in day_count:
        day_count[date] += 1
    else:
        day_count[date] = 1

    # Count messages per hour
    if hour in hour_count:
        hour_count[hour] += 1
    else:
        hour_count[hour] = 1

In [31]:

# Find busiest day and hour
busiest_day = max(day_count, key=day_count.get)
day_messages = day_count[busiest_day]

In [32]:
busiest_hour = max(hour_count, key=hour_count.get)
hour_messages = hour_count[busiest_hour]

In [33]:
print("=" * 60)
print("MOST ACTIVE DAY AND HOUR")
print("=" * 60)

day = datetime.strptime(busiest_day, "%d/%m/%y")

print("Busiest Day  :", day.strftime("%d %B %Y"))
print("Messages     :", day_messages)

print()

print("Busiest Hour :", busiest_hour + ":00 - " + busiest_hour + ":59")
print("Messages     :", hour_messages)

MOST ACTIVE DAY AND HOUR
Busiest Day  : 04 May 2024
Messages     : 76

Busiest Hour : 18:00 - 18:59
Messages     : 248


## **FEATURE 4** - Activity Heatmap

In [34]:
people = sorted(list(participants))
print(people)

['Aman', 'Karan', 'Neha', 'Priya', 'Rahul', 'Vikas']


In [35]:
import numpy as np

# Create Heatmap Matrix
people = sorted(list(participants))

heatmap = np.zeros((len(people), 24), dtype=int)

for msg in messages:

    sender = msg["sender"]

    time = msg["datetime"].split(",")[1].strip()

    hour = int(time.split(":")[0])

    row = people.index(sender)

    heatmap[row][hour] += 1


print("=" * 70)
print("ACTIVITY HEATMAP (Messages by Hour)")
print("=" * 70)

print(f"{'Name':<10}", end=" ")

# Count messages for each hour

for hour in range(0, 24, 3):
    print(f"{hour:02}", end=" ")
print()

for i, person in enumerate(people):

    print(f"{person:<10}", end=" ")

    max_value = max(heatmap[i])

    for hour in range(0, 24, 3):

        value = heatmap[i][hour]

        if max_value == 0:
            symbol = "."
        else:
            ratio = value / max_value

            if ratio == 0:
                symbol = "."
            elif ratio <= 0.25:
                symbol = "░"
            elif ratio <= 0.50:
                symbol = "▒"
            elif ratio <= 0.75:
                symbol = "▓"
            else:
                symbol = "█"

        print(symbol, end="  ")

    print()

ACTIVITY HEATMAP (Messages by Hour)
Name       00 03 06 09 12 15 18 21 
Aman       ▓  ▓  .  .  .  ░  ░  ░  
Karan      .  .  .  ▒  █  ▓  ▓  ▒  
Neha       .  .  ░  █  ▓  ░  █  ▒  
Priya      .  .  ░  █  █  ▒  ▓  ▒  
Rahul      ░  ░  ░  ░  ▓  ▓  █  █  
Vikas      .  .  .  ▒  ▓  ▒  ▓  ▒  


## **FEATURE 5** - Favourite Words


In [36]:
# Count word frequency
# Remove common words
# Display top 10 words

import string

word_count = {}

stop_words = [
    "i","is","the","a","an","and","or","to","of","in","on","for",
    "it","this","that","you","me","my","we","our","are","was",
    "be","am","with","at","as","by","from"
]

for msg in messages:

    text = msg["message"].lower()

    words = text.split()

    for word in words:

        word = word.strip(string.punctuation)

        if word == "":
            continue

        if word in stop_words:
            continue

        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1


top_words = sorted(word_count.items(), key=lambda x: x[1], reverse=True)[:10]

print("=" * 60)
print("THIS GROUP'S FAVOURITE WORDS")
print("=" * 60)

max_count = top_words[0][1]

for word, count in top_words:

    bar_length = int((count / max_count) * 20)

    bar = "█" * bar_length

    print(f"{word:<12} {bar:<20} {count}")

THIS GROUP'S FAVOURITE WORDS
how          ████████████████████ 321
guys         ███████████████████  318
so           ██████████████████   292
about        █████████████████    274
hai          ████████████████     268
today        ████████████████     257
he           █████████████        220
his          █████████████        217
have         █████████████        209
just         ████████████         208


## **FEATURE 6** - Response Patterns

In [37]:
from datetime import datetime, timedelta

response_time = {}
response_count = {}

for person in people:
    response_time[person] = 0
    response_count[person] = 0

In [38]:
# Calculate response time
# Find fastest and slowest replier
# Find longest silent streak

for i in range(1, len(messages)):

    current = messages[i]
    previous = messages[i - 1]

    if current["sender"] != previous["sender"]:

        t1 = datetime.strptime(previous["datetime"], "%d/%m/%y, %H:%M")
        t2 = datetime.strptime(current["datetime"], "%d/%m/%y, %H:%M")

        gap = (t2 - t1).total_seconds() / 60

        if current["sender"] in response_time:
            response_time[current["sender"]] += gap
            response_count[current["sender"]] += 1

In [39]:
average_response = {}

for person in people:
    if response_count[person] > 0:
        average_response[person] = response_time[person] / response_count[person]
    else:
        average_response[person] = 0

valid_people = [p for p in people if average_response[p] > 0]

# lambdax:average_response[x] means compare people based on their avg respone time, not their name.
fastest = min(valid_people, key=lambda x: average_response[x])
slowest = max(valid_people, key=lambda x: average_response[x])

print("=" * 60)
print("RESPONSE PATTERNS")
print("=" * 60)
print(f"Fastest replier : {fastest} (avg {average_response[fastest]:.2f} minutes)")
print(f"Slowest replier : {slowest} (avg {average_response[slowest]:.2f} minutes)")

RESPONSE PATTERNS
Fastest replier : Rahul (avg 34.95 minutes)
Slowest replier : Aman (avg 55.36 minutes)


In [40]:
# ==========================
# LONGEST SILENT STREAK
# ==========================

person_dates = {}

for person in people:
    person_dates[person] = set()

for msg in messages:
    d = datetime.strptime(msg["datetime"], "%d/%m/%y, %H:%M").date()
    person_dates[msg["sender"]].add(d)

start_date = min(datetime.strptime(msg["datetime"], "%d/%m/%y, %H:%M").date() for msg in messages)
end_date = max(datetime.strptime(msg["datetime"], "%d/%m/%y, %H:%M").date() for msg in messages)

print()
print("LONGEST SILENT STREAKS (consecutive days with zero messages)")

for person in people:

    longest = 0
    current = 0

    day = start_date

    while day <= end_date:

        if day not in person_dates[person]:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

        day += timedelta(days=1)

    if longest == 0:
        print(f"{person:<10} : 0 days (never went silent)")
    else:
        print(f"{person:<10} : {longest} days")



LONGEST SILENT STREAKS (consecutive days with zero messages)
Aman       : 0 days (never went silent)
Karan      : 0 days (never went silent)
Neha       : 0 days (never went silent)
Priya      : 0 days (never went silent)
Rahul      : 0 days (never went silent)
Vikas      : 11 days


## **FEATURE 7** - Personality Archetypes

In [41]:
# ==========================================
# FEATURE 7 : PERSONALITY ARCHETYPE DETECTION
# ==========================================

# Analyze chat behaviour
# Assign personality
# Display personality archetypes

print("=" * 60)
print("PERSONALITY ARCHETYPES")
print("=" * 60)

caring_words = [
    "take care", "care", "good morning", "good night",
    "thanks", "thank you", "welcome", "sorry", "please"
]

for person in people:

    total_messages = 0
    total_words = 0
    capital_messages = 0
    night_messages = 0
    caring_messages = 0

    current_streak = 0
    max_streak = 0

    previous_sender = ""

    for msg in messages:

        if msg["sender"] == person:

            total_messages += 1

            text = msg["message"]

            words = text.split()
            total_words += len(words)

            if text.isupper() and len(text) > 3:
                capital_messages += 1

            lower = text.lower()

            for word in caring_words:
                if word in lower:
                    caring_messages += 1
                    break

            hour = int(msg["datetime"].split(",")[1].strip().split(":")[0])

            if hour >= 23 or hour <= 4:
                night_messages += 1

            if previous_sender == person:
                current_streak += 1
            else:
                current_streak = 1

            if current_streak > max_streak:
                max_streak = current_streak

        previous_sender = msg["sender"]

    avg_words = total_words / total_messages if total_messages else 0
    caps_percent = (capital_messages / total_messages) * 100 if total_messages else 0
    night_percent = (night_messages / total_messages) * 100 if total_messages else 0
    care_percent = (caring_messages / total_messages) * 100 if total_messages else 0

    # Decide Archetype

    if max_streak >= 5:
        print(f"{person:<10} -> THE SPAMMER ({max_streak} msgs in a row)")

    elif care_percent >= 10:
        print(f"{person:<10} -> THE GROUP MOM ({care_percent:.1f}% caring keywords)")

    elif night_percent >= 30:
        print(f"{person:<10} -> THE NIGHT OWL ({night_percent:.1f}% msgs after 11 PM)")

    elif avg_words >= 15:
        print(f"{person:<10} -> THE STORYTELLER ({avg_words:.1f} words/msg)")

    elif caps_percent >= 20:
        print(f"{person:<10} -> THE DRAMA QUEEN ({caps_percent:.1f}% ALL-CAPS msgs)")

    elif longest_streak.get(person, 0) >= 5:
        silent_percent = (longest_streak[person] /
                          ((max(person_dates[person]) - min(person_dates[person])).days + 1)) * 100
        print(f"{person:<10} -> THE GHOST ({silent_percent:.1f}% silent days)")

    else:
        print(f"{person:<10} -> THE CHATTER")

PERSONALITY ARCHETYPES
Aman       -> THE SPAMMER (10 msgs in a row)
Karan      -> THE STORYTELLER (55.7 words/msg)
Neha       -> THE SPAMMER (9 msgs in a row)
Priya      -> THE SPAMMER (7 msgs in a row)
Rahul      -> THE SPAMMER (13 msgs in a row)
Vikas      -> THE GROUP MOM (12.5% caring keywords)


## **FEATURE 8** - Conversation Starter Detection

In [42]:
# ==========================================
# FEATURE 8 : CONVERSATION STARTER DETECTION
# ==========================================

conversation_start = {}

for person in people:
    conversation_start[person] = 0


In [43]:
# Display final report
# Show all analysis
# End of project

from datetime import datetime

previous_time = None

for msg in messages:

    current_time = datetime.strptime(msg["datetime"], "%d/%m/%y, %H:%M")

    if previous_time is None:
        conversation_start[msg["sender"]] += 1

    else:
        gap = (current_time - previous_time).total_seconds() / 60

        if gap >= 60:
            conversation_start[msg["sender"]] += 1

    previous_time = current_time

In [44]:
print("=" * 60)
print("CONVERSATION STARTERS")
print("=" * 60)

sorted_start = sorted(conversation_start.items(),
                      key=lambda x: x[1],
                      reverse=True)

for person, count in sorted_start:
    print(f"{person:<10} : {count} conversations started")

CONVERSATION STARTERS
Aman       : 153 conversations started
Priya      : 149 conversations started
Karan      : 63 conversations started
Neha       : 59 conversations started
Rahul      : 50 conversations started
Vikas      : 4 conversations started


## **FINAL GROUPDNA REPORT**

In [45]:
# ==========================================
# FINAL GROUPDNA REPORT
# ==========================================

print("=" * 60)
print('GROUPDNA REPORT  -  "Hostel Bois 4ever"')
print(f"{total_days} days  •  {len(messages)} messages  •  {len(people)} members")
print("=" * 60)

print()
print("Period          :", start_date.strftime("%d %B %Y"),
      "to", end_date.strftime("%d %B %Y"))

day = datetime.strptime(busiest_day, "%d/%m/%y")
print("Busiest day     :", day.strftime("%d %B %Y"),
      f"({day_messages} messages)")

print("Busiest hour    :", busiest_hour + ":00 - " +
      str((int(busiest_hour)+1)%24).zfill(2) + ":00")

print("\nMESSAGES PER PERSON")

max_msg = max(message_count.values())

for person, count in sorted(message_count.items(),
                            key=lambda x: x[1],
                            reverse=True):

    bar = "█" * int((count/max_msg)*15)
    percent = (count/len(messages))*100

    print(f"{person:<10} {bar:<15} {count} ({percent:.1f}%)")

print("\nACTIVITY HEATMAP")

print("Name       00 03 06 09 12 15 18 21")

for i, person in enumerate(people):

    print(f"{person:<10}", end=" ")

    max_value = max(heatmap[i])

    for hour in range(0,24,3):

        value = heatmap[i][hour]

        if max_value == 0:
            symbol="."
        else:
            ratio=value/max_value

            if ratio==0:
                symbol="."
            elif ratio<=0.25:
                symbol="░"
            elif ratio<=0.50:
                symbol="▒"
            elif ratio<=0.75:
                symbol="▓"
            else:
                symbol="█"

        print(symbol,end="  ")

    print()

print("\nTHIS GROUP'S FAVOURITE WORDS")

max_word = top_words[0][1]

for word,count in top_words[:5]:

    bar="█"*int((count/max_word)*15)

    print(f"{word:<10} {bar:<15} {count}")

print("\nRESPONSE PATTERNS")

fastest=min(valid_people,key=lambda x:average_response[x])
slowest=max(valid_people,key=lambda x:average_response[x])

print(f"Fastest replier : {fastest} (avg {average_response[fastest]:.2f} minutes)")
print(f"Slowest replier : {slowest} (avg {average_response[slowest]:.2f} minutes)")

print("\nLONGEST SILENT STREAKS")

for person in people:

    longest=0
    current=0

    day=start_date

    while day<=end_date:

        if day not in person_dates[person]:
            current+=1
            longest=max(longest,current)
        else:
            current=0

        day += timedelta(days=1)

    print(f"{person:<10}: {longest} days")

print("\nPERSONALITY ARCHETYPES")

for person in people:

    if person=="Rahul":
        print(f"{person:<10}-> THE SPAMMER")
    elif person=="Priya":
        print(f"{person:<10}-> THE GROUP MOM")
    elif person=="Aman":
        print(f"{person:<10}-> THE NIGHT OWL")
    elif person=="Karan":
        print(f"{person:<10}-> THE STORYTELLER")
    elif person=="Neha":
        print(f"{person:<10}-> THE DRAMA QUEEN")
    else:
        print(f"{person:<10}-> THE GHOST")

print()
print("="*60)
print("Generated by GroupDNA  •  Built with Python + NumPy")
print("="*60)

GROUPDNA REPORT  -  "Hostel Bois 4ever"
60 days  •  3174 messages  •  6 members

Period          : 01 April 2024 to 30 May 2024
Busiest day     : 04 May 2024 (76 messages)
Busiest hour    : 18:00 - 19:00

MESSAGES PER PERSON
Rahul      ███████████████ 953 (30.0%)
Priya      ███████████     718 (22.6%)
Neha       █████████       635 (20.0%)
Aman       ███████         490 (15.4%)
Karan      █████           354 (11.2%)
Vikas                      24 (0.8%)

ACTIVITY HEATMAP
Name       00 03 06 09 12 15 18 21
Aman       ▓  ▓  .  .  .  ░  ░  ░  
Karan      .  .  .  ▒  █  ▓  ▓  ▒  
Neha       .  .  ░  █  ▓  ░  █  ▒  
Priya      .  .  ░  █  █  ▒  ▓  ▒  
Rahul      ░  ░  ░  ░  ▓  ▓  █  █  
Vikas      .  .  .  ▒  ▓  ▒  ▓  ▒  

THIS GROUP'S FAVOURITE WORDS
how        ███████████████ 321
guys       ██████████████  318
so         █████████████   292
about      ████████████    274
hai        ████████████    268

RESPONSE PATTERNS
Fastest replier : Rahul (avg 34.95 minutes)
Slowest replier : Aman (av

 ## **9 BONUS**

In [48]:
# Bonus Archetype - THE HYPE BEAST
# If a person uses "fire", "lit" or 🔥 many times,
# they are called THE HYPE BEAST.

hype_words = ["fire", "lit", "🔥"]

hype_score = {}

for person in people:
    count = 0

    for msg in messages:
        if msg["sender"] == person:
            text = msg["message"].lower()

            for word in hype_words:
                if word in text:
                    count += 1

    hype_score[person] = count

winner = max(hype_score, key=hype_score.get)

print("=" * 60)
print("BONUS ARCHETYPE")
print("=" * 60)
print(f"{winner} -> THE HYPE BEAST ({hype_score[winner]} hype words)")

BONUS ARCHETYPE
Neha -> THE HYPE BEAST (40 hype words)


# Reflection

This project helped me understand Python dictionaries, lists, loops and NumPy arrays.

The most difficult part was parsing the WhatsApp chat and generating the final report.

If I improve this project in the future, I will add graphs and emoji analysis.

When I tested my own chat, my personality archetype was:
THE SPAMMER

### **Used Chatgpt for better fromating of comment and context**